# Day 16: Convolution — The Core Operation

> **Goal:** Deeply understand the convolution operation — how it works, its parameters (kernel, stride, padding), and why it's perfect for images.

---

## 🔬 Optics Connection

Convolution should feel familiar! In optics:
- The **Point Spread Function (PSF)** of a lens convolves with the object to form the image
- Image filtering (blur, sharpen, edge detect) = convolution with a kernel

In deep learning, the **kernel weights are learned** — the network discovers its own optimal filters!

## Why Convolution for Images?

An MLP flattens images, destroying spatial structure. Convolutions exploit:

1. **Local connectivity** — each output only depends on a small neighbourhood
2. **Weight sharing** — the same filter slides across the entire image
3. **Translation equivariance** — a feature is detected regardless of position
4. **Far fewer parameters** — 3×3 kernel = 9 weights vs millions for dense layers

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

torch.manual_seed(42)

---
## 1. Convolution by Hand

A 2D convolution slides a small **kernel** across the image. At each position, it computes a dot product (element-wise multiply + sum).

In [ ]:
# Manual 2D convolution (no PyTorch)
def manual_conv2d(image, kernel):
    """Simple 2D convolution (no padding, stride=1)."""
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = h - kh + 1
    out_w = w - kw + 1
    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(patch * kernel)

    return output

# Small test
image = np.array([
    [1, 2, 3, 0],
    [0, 1, 2, 3],
    [3, 0, 1, 2],
    [2, 3, 0, 1],
], dtype=np.float32)

# Edge detection kernel
kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=np.float32)

result = manual_conv2d(image, kernel)
print("Input (4×4):")
print(image)
print(f"\nKernel (3×3):")
print(kernel)
print(f"\nOutput (2×2):")
print(result)

---
## 2. Classical Image Filters as Convolutions

These are the same convolution kernels used in image processing — but in a CNN, the network **learns** them!

In [ ]:
# Create a simple test image
from torchvision.datasets import CIFAR10
import torchvision.transforms as T

cifar = CIFAR10(root="./data", train=True, download=True, transform=T.ToTensor())
img_tensor, _ = cifar[42]  # (3, 32, 32)

# Convert to grayscale for clearer visualization
gray = img_tensor.mean(dim=0, keepdim=True).unsqueeze(0)  # (1, 1, 32, 32)

# Define classical kernels
kernels = {
    "Identity": torch.tensor([[0,0,0],[0,1,0],[0,0,0]], dtype=torch.float32),
    "Blur (box)": torch.ones(3,3) / 9,
    "Sharpen": torch.tensor([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=torch.float32),
    "Edge (Laplacian)": torch.tensor([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]], dtype=torch.float32),
    "Sobel X": torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32),
    "Sobel Y": torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, k) in zip(axes.flat, kernels.items()):
    # Apply convolution using F.conv2d
    k4d = k.view(1, 1, 3, 3)  # (out_channels, in_channels, kH, kW)
    result = F.conv2d(gray, k4d, padding=1)
    ax.imshow(result.squeeze().detach().numpy(), cmap="gray")
    ax.set_title(name, fontsize=12)
    ax.axis("off")

plt.suptitle("Classical Convolution Kernels Applied to CIFAR Image", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. `nn.Conv2d` — PyTorch's Convolution Layer

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

| Parameter | Meaning |
|---|---|
| `in_channels` | Number of input channels (1 for gray, 3 for RGB) |
| `out_channels` | Number of filters (= number of output channels) |
| `kernel_size` | Size of the filter (e.g., 3 means 3×3) |
| `stride` | Step size when sliding the kernel |
| `padding` | Zeros added to input borders |

In [ ]:
# Create a Conv2d layer
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

print(f"Weight shape: {conv.weight.shape}")  # (out_ch, in_ch, kH, kW)
print(f"Bias shape:   {conv.bias.shape}")
print(f"Parameters:   {conv.weight.numel() + conv.bias.numel()}")
print(f"              = {3*3*3*16} (weight) + {16} (bias) = {3*3*3*16+16}")

# Forward pass
x = torch.randn(1, 3, 32, 32)  # (batch, channels, height, width)
y = conv(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {y.shape}")  # (1, 16, 32, 32) — same spatial size with padding=1

---
## 4. Understanding Stride and Padding

### Output size formula:

$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2P - K}{S} \right\rfloor + 1$$

where P = padding, K = kernel size, S = stride.

In [ ]:
def show_conv_output_size(in_size, kernel_size, stride, padding):
    out_size = (in_size + 2 * padding - kernel_size) // stride + 1
    print(f"Input: {in_size}×{in_size}, Kernel: {kernel_size}, "
          f"Stride: {stride}, Padding: {padding} → Output: {out_size}×{out_size}")
    return out_size

print("=== Common Configurations ===")
show_conv_output_size(32, 3, 1, 0)  # 32 → 30 (shrinks!)
show_conv_output_size(32, 3, 1, 1)  # 32 → 32 (same size - "same" padding)
show_conv_output_size(32, 3, 2, 1)  # 32 → 16 (halved - stride 2)
show_conv_output_size(32, 5, 1, 2)  # 32 → 32 (5×5 kernel, same size)
show_conv_output_size(32, 7, 1, 3)  # 32 → 32 (7×7 kernel, same size)

### Verification with Real Layers

Let's verify our formula by running actual Conv2d layers and checking the output sizes match:

In [ ]:
# Verify with actual layers
x = torch.randn(1, 3, 32, 32)

configs = [
    {"kernel_size": 3, "stride": 1, "padding": 0},
    {"kernel_size": 3, "stride": 1, "padding": 1},
    {"kernel_size": 3, "stride": 2, "padding": 1},
    {"kernel_size": 5, "stride": 1, "padding": 2},
]

for cfg in configs:
    conv = nn.Conv2d(3, 16, **cfg)
    out = conv(x)
    print(f"kernel={cfg['kernel_size']}, stride={cfg['stride']}, "
          f"pad={cfg['padding']} → {out.shape[2]}×{out.shape[3]}")

---
## 5. Pooling Layers

**Pooling** reduces spatial dimensions (downsampling).

- `nn.MaxPool2d(2)` — takes the maximum in each 2×2 window → halves spatial size
- `nn.AvgPool2d(2)` — takes the average
- `nn.AdaptiveAvgPool2d(1)` — average pool to a specific output size (used before FC layers)

In [ ]:
x = torch.randn(1, 16, 32, 32)

# MaxPool
pool = nn.MaxPool2d(kernel_size=2, stride=2)
print(f"MaxPool2d(2): {x.shape} → {pool(x).shape}")  # 32 → 16

# Adaptive pool to 1×1 (global average pooling)
gap = nn.AdaptiveAvgPool2d(1)
print(f"AdaptiveAvgPool2d(1): {x.shape} → {gap(x).shape}")  # → (1, 16, 1, 1)

# Visualize max pooling
small = torch.tensor([[1, 2, 3, 4],
                       [5, 6, 7, 8],
                       [9, 10, 11, 12],
                       [13, 14, 15, 16]], dtype=torch.float32)

small_4d = small.unsqueeze(0).unsqueeze(0)  # (1, 1, 4, 4)
pooled = nn.MaxPool2d(2)(small_4d)

print(f"\nInput:\n{small.numpy()}")
print(f"\nMaxPool2d(2) output:\n{pooled.squeeze().numpy()}")

---
## 6. Visualizing Learned Features

Let's see what conv layer outputs ("feature maps") look like.

In [ ]:
# Apply a randomly initialized conv to a CIFAR image
conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)

img, label = cifar[42]  # (3, 32, 32)
img_batch = img.unsqueeze(0)  # (1, 3, 32, 32)

with torch.no_grad():
    features = conv1(img_batch)  # (1, 8, 32, 32)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
# Original
axes[0, 0].imshow(img.permute(1, 2, 0))
axes[0, 0].set_title("Original")

# Feature maps
for i in range(1, 10):
    r, c = divmod(i, 5)
    fmap = features[0, i-1].detach().numpy()
    axes[r, c].imshow(fmap, cmap="viridis")
    axes[r, c].set_title(f"Filter {i}")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Feature Maps from Random Conv Layer", fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Parameter Comparison: Conv vs Dense

This is a key advantage of convolutions:

In [ ]:
# Processing a 32×32×3 image:
# Dense layer to produce 16 features from each pixel
dense = nn.Linear(32 * 32 * 3, 32 * 32 * 16)
conv  = nn.Conv2d(3, 16, kernel_size=3, padding=1)

dense_params = sum(p.numel() for p in dense.parameters())
conv_params  = sum(p.numel() for p in conv.parameters())

print(f"Dense layer parameters: {dense_params:>12,}")
print(f"Conv layer parameters:  {conv_params:>12,}")
print(f"\nConv is {dense_params / conv_params:.0f}× fewer parameters!")

---
## 🧪 Exercises

### Exercise 1: Manual convolution
Apply the Sobel-X and Sobel-Y kernels to a CIFAR image manually (using numpy). Compute the gradient magnitude: `sqrt(Gx² + Gy²)`. Display the result.

### Exercise 2: Output size calculator
Write a function that takes input size, a list of conv configs (kernel, stride, padding), and computes the output size after each layer.

### Exercise 3: 1×1 convolutions
`Conv2d(in_ch, out_ch, kernel_size=1)` doesn't look at spatial neighbours — it mixes channels. Apply a 1×1 conv to reduce 16 channels to 4. How many parameters does it have?

### Exercise 4: Receptive field
Stack 3 Conv2d(3, padding=1) layers. What is the effective receptive field? (Hint: after 1 layer it's 3×3, after 2 layers it's 5×5...)

In [ ]:
# Exercise 1: Your code here

In [ ]:
# Exercise 2: Your code here

In [ ]:
# Exercise 3: Your code here

In [ ]:
# Exercise 4: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Convolution** | Slide a kernel across the image, compute dot products |
| **nn.Conv2d** | `(in_ch, out_ch, kernel_size, stride, padding)` |
| **Padding** | "same" padding = `kernel_size // 2` to preserve spatial size |
| **Stride** | Stride 2 halves the spatial dimensions |
| **Pooling** | MaxPool2d / AvgPool2d reduce spatial size |
| **Weight sharing** | Same filter used everywhere → far fewer parameters than dense |
| **Feature maps** | Each output channel = response to one learned filter |

**Tomorrow:** We'll build our first complete CNN and train it on CIFAR-10!